# 03 — Analysis


## Cell 1 — Connect

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

with engine.connect() as conn:
    print("Connected:", conn.execute(text("SELECT version();")).fetchone()[0][:30])


Connected: PostgreSQL 18.4 on x86_64-wind


## Q1 — Is the business growing, and is profit keeping pace with sales? 


In [2]:
query = '''
WITH yearly AS (
    SELECT EXTRACT(YEAR FROM order_date) AS year,
           SUM(sales) AS total_sales,
           SUM(profit) AS total_profit
    FROM fact_order_lines
    GROUP BY EXTRACT(YEAR FROM order_date)
)
SELECT year,
       ROUND(total_sales, 0) AS total_sales,
       ROUND(total_profit, 0) AS total_profit,
       ROUND(100.0 * (total_sales - LAG(total_sales) OVER (ORDER BY year)) / LAG(total_sales) OVER (ORDER BY year), 1) AS sales_yoy_pct,
       ROUND(100.0 * (total_profit - LAG(total_profit) OVER (ORDER BY year)) / LAG(total_profit) OVER (ORDER BY year), 1) AS profit_yoy_pct
FROM yearly
ORDER BY year;
'''
yoy_df = pd.read_sql(text(query), engine)
yoy_df


,year,total_sales,total_profit,sales_yoy_pct,profit_yoy_pct
0,2012.0,2259451.0,248941.0,NaN,NaN
1,2013.0,2677439.0,307415.0,18.5,23.5
2,2014.0,3405746.0,406935.0,27.2,32.4
3,2015.0,4299866.0,504166.0,26.3,23.9


## Q2 — Which sub-categories generate sales but destroy profit? 

In [3]:
query = '''
SELECT p.sub_category,
       COUNT(*) AS line_count,
       ROUND(SUM(f.sales)::numeric, 0) AS total_sales,
       ROUND(SUM(f.profit)::numeric, 0) AS total_profit,
       ROUND(100.0 * SUM(f.profit) / NULLIF(SUM(f.sales), 0), 1) AS margin_pct,
       ROUND(100.0 * AVG(CASE WHEN f.profit < 0 THEN 1 ELSE 0 END), 1) AS pct_loss_making_lines
FROM fact_order_lines f
JOIN dim_product p ON f.product_id = p.product_id
GROUP BY p.sub_category
ORDER BY total_profit ASC;
'''
profit_leakage_df = pd.read_sql(text(query), engine)
profit_leakage_df


,sub_category,line_count,total_sales,total_profit,margin_pct,pct_loss_making_lines
0,Tables,861,757042.0,-64083.0,-8.5,57.6
1,Fasteners,2601,89495.0,13844.0,15.5,24.1
2,Labels,2601,73350.0,14989.0,20.4,20.0
3,Supplies,2407,242811.0,22559.0,9.3,24.1
4,Envelopes,2387,169217.0,28849.0,17.0,23.3
5,Furnishings,3154,385156.0,46845.0,12.2,25.3
6,Art,4864,371613.0,57830.0,15.6,18.7
7,Paper,3492,241788.0,58112.0,24.0,15.1
8,Machines,1486,779060.0,58868.0,7.6,30.6
9,Binders,6146,461869.0,72433.0,15.7,24.9


## Q3 — At what discount level does profitability deteriorate? 

In [4]:
query = '''
SELECT p.sub_category,
       CASE
           WHEN f.discount = 0 THEN '0%'
           WHEN f.discount <= 0.2 THEN '0-20%'
           WHEN f.discount <= 0.4 THEN '20-40%'
           WHEN f.discount <= 0.6 THEN '40-60%'
           ELSE '60%+'
       END AS discount_band,
       COUNT(*) AS line_count,
       ROUND(100.0 * SUM(f.profit) / NULLIF(SUM(f.sales), 0), 1) AS overall_margin_pct
FROM fact_order_lines f
JOIN dim_product p ON f.product_id = p.product_id
WHERE p.sub_category IN ('Tables', 'Bookcases', 'Chairs')
GROUP BY p.sub_category, discount_band
ORDER BY p.sub_category, MIN(f.discount);
'''
discount_band_df = pd.read_sql(text(query), engine)
discount_band_df


,sub_category,discount_band,line_count,overall_margin_pct
0,Bookcases,0%,1127,24.4
1,Bookcases,0-20%,712,13.1
2,Bookcases,20-40%,307,-23.7
3,Bookcases,40-60%,190,-65.4
4,Bookcases,60%+,75,-152.0
5,Chairs,0%,1336,23.9
6,Chairs,0-20%,1179,10.6
7,Chairs,20-40%,638,-10.3
8,Chairs,40-60%,227,-72.8
9,Chairs,60%+,54,-134.2


## Q4 — Which markets grew, and did profitability improve or deteriorate? 

In [5]:
query = '''
WITH market_year AS (
    SELECT g.market, EXTRACT(YEAR FROM f.order_date) AS year,
           SUM(f.sales) AS sales,
           SUM(f.profit) AS profit
    FROM fact_order_lines f
    JOIN dim_geography g ON f.geography_key = g.geography_key
    GROUP BY g.market, EXTRACT(YEAR FROM f.order_date)
),
first_last AS (
    SELECT market, MIN(year) AS first_year, MAX(year) AS last_year
    FROM market_year GROUP BY market
)
SELECT fl.market,
       ROUND(my_first.sales::numeric, 0) AS sales_2012,
       ROUND(my_last.sales::numeric, 0) AS sales_2015,
       ROUND(100.0 * (my_last.sales - my_first.sales) / my_first.sales, 1) AS sales_growth_pct,
       ROUND(100.0 * my_first.profit / my_first.sales, 1) AS margin_pct_2012,
       ROUND(100.0 * my_last.profit / my_last.sales, 1) AS margin_pct_2015
FROM first_last fl
JOIN market_year my_first ON my_first.market = fl.market AND my_first.year = fl.first_year
JOIN market_year my_last ON my_last.market = fl.market AND my_last.year = fl.last_year
ORDER BY sales_growth_pct DESC;
'''
market_trend_df = pd.read_sql(text(query), engine)
market_trend_df


,market,sales_2012,sales_2015,sales_growth_pct,margin_pct_2012,margin_pct_2015
0,Africa,127187.0,283036.0,122.5,8.6,13.9
1,Europe,540751.0,1180304.0,118.3,14.2,13.3
2,Asia Pacific,713658.0,1372784.0,92.4,10.3,9.8
3,LATAM,385098.0,706633.0,83.5,9.5,10.4
4,USCA,492757.0,757108.0,53.6,10.4,13.1


## Q5 — How dependent is each market on a small number of customers? 

In [6]:
query = '''
WITH cust_profit AS (
    SELECT g.market, c.customer_id, c.customer_name, SUM(f.profit) AS customer_profit
    FROM fact_order_lines f
    JOIN dim_customer c ON f.customer_id = c.customer_id
    JOIN dim_geography g ON f.geography_key = g.geography_key
    GROUP BY g.market, c.customer_id, c.customer_name
),
market_totals AS (
    SELECT market, SUM(customer_profit) AS market_total_profit
    FROM cust_profit GROUP BY market
),
ranked AS (
    SELECT cp.*, mt.market_total_profit,
           RANK() OVER (PARTITION BY cp.market ORDER BY customer_profit DESC) AS rnk
    FROM cust_profit cp
    JOIN market_totals mt ON cp.market = mt.market
)
SELECT market, customer_id, customer_name,
       ROUND(customer_profit::numeric, 0) AS customer_profit,
       ROUND(100.0 * customer_profit / NULLIF(market_total_profit, 0), 1) AS pct_of_market_profit,
       rnk
FROM ranked
WHERE rnk <= 5
ORDER BY market, rnk;
'''
top_customers_df = pd.read_sql(text(query), engine)
top_customers_df


,market,customer_id,customer_name,customer_profit,pct_of_market_profit,rnk
0,Africa,BW-106533,Barry Weirich,3262.0,3.7,1
1,Africa,DP-310586,Dave Poirier,2625.0,3.0,2
2,Africa,MG-814533,Mike Gockenbach,1869.0,2.1,3
3,Africa,LT-7110117,Liz Thompson,1631.0,1.8,4
4,Africa,JW-5220111,Jane Waco,1601.0,1.8,5
5,Asia Pacific,CA-1277558,Cynthia Arntzen,3982.0,1.0,1
6,Asia Pacific,CA-1196566,Carol Adams,3034.0,0.8,2
7,Asia Pacific,VG-2180558,Vivek Grady,2538.0,0.6,3
8,Asia Pacific,AS-1063092,Ann Steele,2417.0,0.6,4
9,Asia Pacific,DO-1343558,Denny Ordway,2285.0,0.6,5


In [ ]:
# How much of each market's profit sits in just its top 5 customers?
concentration = top_customers_df.groupby("market")["pct_of_market_profit"].sum().reset_index()
concentration.columns = ["market", "top5_pct_of_market_profit"]
concentration.sort_values("top5_pct_of_market_profit", ascending=False)


,market,top5_pct_of_market_profit
0,Africa,12.4
4,USCA,10.1
1,Asia Pacific,3.6
3,LATAM,3.1
2,Europe,3.0


## RFM — Which customer groups represent retention or reactivation opportunities?


In [8]:
query = '''
SELECT customer_id,
       MAX(order_date) AS last_order_date,
       COUNT(DISTINCT order_id) AS frequency,
       SUM(sales) AS monetary,
       SUM(profit) AS total_profit
FROM fact_order_lines
GROUP BY customer_id;
'''
rfm_raw = pd.read_sql(text(query), engine)
rfm_raw["last_order_date"] = pd.to_datetime(rfm_raw["last_order_date"])
print(f"Customers: {len(rfm_raw):,}")
rfm_raw.head()


Customers: 17,415


,customer_id,last_order_date,frequency,monetary,total_profit
0,AA-10315102,2015-01-07,2,544.656,-153.0540
1,AA-10315120,2013-05-16,1,2713.410,27.0900
2,AA-10315139,2015-08-04,4,2955.798,514.6680
3,AA-103151402,2015-06-30,2,4780.552,-650.5971
4,AA-103151404,2013-10-04,2,753.508,274.4320


In [9]:
snapshot_date = rfm_raw["last_order_date"].max() + pd.Timedelta(days=1)
rfm_raw["recency_days"] = (snapshot_date - rfm_raw["last_order_date"]).dt.days

# R and M genuinely split into 5 even groups. F does not — frequency is
# concentrated at 1-2 orders, so it gets 4 fixed tiers instead of a forced
# 5-way qcut that would just create meaningless ties. This is stated here,
# not hidden behind a label that implies false symmetry.
rfm_raw["R_score"] = pd.qcut(rfm_raw["recency_days"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm_raw["M_score"] = pd.qcut(rfm_raw["monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm_raw["F_score"] = pd.cut(rfm_raw["frequency"], bins=[0, 1, 2, 3, 100], labels=[1, 2, 3, 4]).astype(int)

def segment(row):
    if row["R_score"] >= 4 and row["F_score"] >= 3:
        return "Champions"
    if row["R_score"] >= 4 and row["F_score"] <= 2:
        return "Recent, Low Frequency"
    if row["R_score"] <= 2 and row["F_score"] >= 3:
        return "At Risk (Previously Frequent)"
    if row["R_score"] <= 2 and row["M_score"] >= 4:
        return "Lapsed High-Value"
    if row["R_score"] <= 2 and row["F_score"] <= 2:
        return "Lost / Low Value"
    return "Regular"

rfm_raw["segment"] = rfm_raw.apply(segment, axis=1)

summary = rfm_raw.groupby("segment").agg(
    n_customers=("customer_id", "count"),
    total_monetary=("monetary", "sum"),
    total_profit=("total_profit", "sum"),
).reset_index()
summary["pct_of_customers"] = round(summary["n_customers"] / len(rfm_raw) * 100, 1)
summary["pct_of_total_profit"] = round(summary["total_profit"] / rfm_raw["total_profit"].sum() * 100, 1)
summary.sort_values("total_profit", ascending=False)


,segment,n_customers,total_monetary,total_profit,pct_of_customers,pct_of_total_profit
4,"Recent, Low Frequency",5575,3.568454e+06,395087.79998,32.0,26.9
2,Lapsed High-Value,2132,2.908446e+06,377516.60748,12.2,25.7
1,Champions,1396,2.518783e+06,355167.25480,8.0,24.2
5,Regular,3483,2.575390e+06,294862.17102,20.0,20.1
0,At Risk (Previously Frequent),192,3.302804e+05,34479.06632,1.1,2.3
3,Lost / Low Value,4637,7.411479e+05,10344.39168,26.6,0.7


In [10]:
rfm_raw.to_sql("rfm_customers", engine, if_exists="replace", index=False)
summary.to_sql("rfm_segment_summary", engine, if_exists="replace", index=False)

print("Written to Postgres: rfm_customers, rfm_segment_summary")


Written to Postgres: rfm_customers, rfm_segment_summary


In [11]:
with engine.connect() as conn:
    for tbl in ["rfm_customers", "rfm_segment_summary"]:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {tbl}")).scalar()
        print(f"{tbl}: {n:,} rows in Postgres")


rfm_customers: 17,415 rows in Postgres
rfm_segment_summary: 6 rows in Postgres
